In [1]:
import json
from pathlib import Path
from prompts.schemas import ContributionType, Area
import pandas as pd

## Load data

In [2]:
data_dir = Path('../data/extracted_data')
filename = 'entities_and_results.json'
with open(data_dir / filename) as f:
    data = json.load(f)

In [3]:
print(f'There are {len(data)} entries in the data.')

There are 49689 entries in the data.


### Make Dataframe for Entities

In [4]:
def standardize_usage(
    usage: list[str]
) -> str:
    for usage_entry in usage:
        if usage_entry not in ['Proposed Model', 'Baseline']:
            print(f'Found unexpected value {usage_entry}.')
            return usage_entry
    if 'Proposed Model' and 'Baseline' in usage:
        return 'Both'
    elif 'Proposed Model' in usage:
        return 'Proposed Model'
    elif 'Baseline' in usage:
        return 'Baseline'
    else:
        return ''

category_names = {
    'tasks': 'task',
    'datasets': 'dataset',
    'metrics': 'metric',
    'architectures': 'architecture',
    'methods': 'method',
    'pretrained_models': 'pretrained_model'
}


# Make dataframe for entities
entities_df_data = {
    'id': [],
    'category': [],
    'name': [],
    'quote': [],
    'usage': []
}

for id, entry in data.items():
    for category, extracted_entities in entry['entities'].items():
        for extracted_entity in extracted_entities:
            name = extracted_entity['name'].strip().lower()
            quote = extracted_entity['quote']
            if 'usage' in extracted_entity:
                usage = standardize_usage(extracted_entity['usage'])
            else:
                usage = ''
            entities_df_data['id'].append(id)
            entities_df_data['category'].append(category_names[category])
            entities_df_data['name'].append(name)
            entities_df_data['quote'].append(quote)
            entities_df_data['usage'].append(usage)

entities_df = pd.DataFrame(entities_df_data)

### Make dataframe for results

In [6]:
results_df_data = {
    'id': [],
    'dataset': [],
    'task': [],
    'metric': [],
    'result': []
}

for id, entry in data.items():
    for result in entry['results']:
        results_df_data['id'].append(id)
        results_df_data['dataset'].append(result['dataset'].strip().lower())
        results_df_data['task'].append(result['task'].strip().lower())
        results_df_data['metric'].append(result['metric'].strip().lower())
        results_df_data['result'].append(result['result'])

results_df = pd.DataFrame(results_df_data)

## Data Analysis

### Functions

### Entities

Get the total number of extracted entities.

In [7]:
print(len(entities_df))

437646


Get number of entries per category and number of unique values

In [8]:
entities_df.groupby('category').agg({'name': ['count', 'nunique']})

name        
                   count nunique
category                        
architecture       72928   18673
dataset           103858   41788
method             84387   36794
metric             81641   11795
pretrained_model   24538    4579
task               70294   21094

Get the 20 most common entities per category.

In [9]:
# Get the 20 most common entities per category and their count
for category in entities_df['category'].unique():
    print(f'Category: {category}')
    print(entities_df.groupby('category').get_group(category)['name'].value_counts().head(20))
    print('\n')

Category: task
name
machine translation                2396
named entity recognition           1763
sentiment analysis                 1645
dependency parsing                  971
question answering                  939
neural machine translation          860
part-of-speech tagging              803
statistical machine translation     731
natural language inference          670
text classification                 652
word sense disambiguation           571
relation extraction                 537
coreference resolution              483
semantic role labeling              403
natural language generation         399
information extraction              382
sentiment classification            378
language modeling                   347
automatic speech recognition        339
semantic parsing                    333
Name: count, dtype: int64


Category: dataset
name
europarl                       924
conll-2003                     733
penn treebank                  667
wikipedia               

### Results

Get the number of extracted results.

In [10]:
print(len(results_df))
results_df.nunique()

86905


id         38209
dataset    28647
task       16055
metric      7928
result     11490
dtype: int64

## Combine entities from Entities and Results

In [11]:
results_as_entity_rows = []
for _, row in results_df.iterrows():
    dataset_row = {
        'id': row['id'],
        'category': 'dataset',
        'name': row['dataset'],
        'quote': '',
        'usage': ''
    }
    task_row = {
        'id': row['id'],
        'category': 'task',
        'name': row['task'],
        'quote': '',
        'usage': ''
    }
    metric_row = {
        'id': row['id'],
        'category': 'metric',
        'name': row['metric'],
        'quote': '',
        'usage': ''
    }
    results_as_entity_rows += [
        dataset_row,
        task_row,
        metric_row
    ]
entities_from_results_df = pd.DataFrame(results_as_entity_rows)

combined_df = pd.concat(
    [
        entities_df,
        entities_from_results_df
    ],
    ignore_index=True
)

Get the number of unique entities in the combined df

In [12]:
combined_df.groupby('category').agg({'name': ['count', 'nunique']})

name        
                   count nunique
category                        
architecture       72928   18673
dataset           190763   46400
method             84387   36794
metric            168546   12844
pretrained_model   24538    4579
task              157199   22292

## Check for duplicates